# 03 — Two-Stage Default · CLF · xgb (raw mode — 전처리 없음)

`zit_only_raw` 패턴을 Stage 1 분류기에 적용. `preprocess.run()` 생략하고 XGBoost의 NaN 네이티브 처리(Sparsity-Aware Algorithm)에 맡겨 원본 분포 그대로 학습. 출력은 die-level prob csv → combine에서 reg와 곱한다.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/clf/xgb_raw/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **전처리**: Stage 0만 적용. cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP
- **HPO**: `N_TRIALS=10000` + `TIMEOUT_SEC=24h` 안전망
- **anchor enqueue 없음**

## 1. 환경 설정 + import

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'   # preprocessing.zip (raw는 EXCLUDE_COLS·meta_features만 사용, import 호환용)
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID        = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'   # 4_output.zip = 기존 실험 산출물 (RESUME 시 복원용)
RESUME                  = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../../../setup.py만 실행
try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    # RESUME이면 이전 4_output을 통째로 복원 (이미 폴더 있으면 skip)
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 추가 — raw는 EXCLUDE_COLS·meta_features만 쓰지만 같은 폴더라 sys.path 추가 필요
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
# `from modules import ...`가 3_modeling/modules를 찾게
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

# raw 모드: preprocess.run()은 사용 안 함 — EXCLUDE_COLS(Stage 0 제외 목록)만 가져옴
from modules.preprocess import EXCLUDE_COLS as _WAFER_MAP_EXCLUDE
from modules import hpo, models                    # hpo.run_clf_hpo / refit_clf_best / save_clf_artifacts, models 레지스트리
from meta_features import add_meta_features         # die_xy / position 메타피처 헬퍼

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)   # LGBM 로그 억제
optuna.logging.set_verbosity(optuna.logging.WARNING)    # Optuna 로그 억제

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

## 2. 실험 설정

In [ ]:
# 분류 모델 고정 (Two-Stage Stage 1 — y>0인 unit 식별, raw mode)
CLF_MODEL_NAME = 'xgb'
assert CLF_MODEL_NAME in models.CLF_AVAILABLE_MODELS

# 실험 식별 — 출력 폴더/DB 파일명에 들어감 (raw임을 명시)
EXP_ID   = f'ts-clf-{CLF_MODEL_NAME}-raw-001'
EXP_MEMO = f'Two-Stage default · CLF · {CLF_MODEL_NAME} · raw mode (preprocess.run 생략, NaN 네이티브 - Sparsity-Aware Algorithm)'
USER     = 'jh'

# Optuna 예산 — 로컬 런스크립트에서 타임아웃으로 컨트롤. trial은 크게, timeout은 안전망
N_TRIALS         = 10000     # 외부 런스크립트 타임아웃이 실제 종료를 결정하므로 충분히 크게
N_FOLDS          = 5
N_STARTUP_TRIALS = 50        # TPE가 학습을 시작하기 전 무작위 trial 수
N_JOBS           = -1        # 모델 학습 병렬도 (-1 = 전체 코어)
TIMEOUT_SEC      = 24 * 60 * 60   # 24h 안전망 — 노트북 단독 실행 시 보호장치

# 출력 경로 — 모델명 뒤에 _raw 표기 (기존 비-raw 디렉토리와 충돌 방지)
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'clf', CLF_MODEL_NAME, 'raw', EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')   # study가 여기에 자동 저장 (RESUME 시 여기서 이어서)
os.makedirs(OUT_DIR, exist_ok=True)

# raw 모드 — PP_FIXED 없음 (cleaning/imputation/outlier 전부 SKIP, Stage 0만 적용)

CLIP_Y_EXTREME = True   # train y의 max(=1.0, 단 1건)를 두 번째 큰 값으로 clip (학습 입력 안정화)

print(f'EXP: {EXP_ID} | USER: {USER} | raw_mode=True')
print(f'CLF_MODEL_NAME: {CLF_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + Y clip + Stage 0 (raw mode)

`preprocess.run()` 호출 없음. cleaning / spatial imputation / winsorize / 고상관 제거 전부 SKIP. XGBoost는 Sparsity-Aware Algorithm으로 NaN을 default direction split으로 학습하므로 imputation 불필요.

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# Stage 0만 적용
n_before = len(feat_cols)
feat_cols_raw = [c for c in feat_cols if c not in _WAFER_MAP_EXCLUDE]
print(f'[Stage 0] 웨이퍼맵 사전 제외: {n_before} → {len(feat_cols_raw)} ({n_before - len(feat_cols_raw)}개 제거)')
print('[raw mode] cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP')

xs_train = xs_dict['train'].copy()
xs_val   = xs_dict['validation'].copy()
xs_test  = xs_dict['test'].copy()

# 메타피처: 트리 분류기도 position raw 정수 + die_x/die_y 연속형
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_raw,
    position_mode='raw', use_die_xy=True,
)

nan_pct_train = xs_train[feat_cols_clean].isna().to_numpy().mean() * 100
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  NaN in train[feat_cols]: {xs_train[feat_cols_clean].isna().to_numpy().sum():,} ({nan_pct_train:.2f}%) — XGBoost가 직접 처리')

# 이진 라벨 클래스 분포
y_train_unit = ys_input['train']
n_pos = (y_train_unit[TARGET_COL] > 0).sum()
n_neg = (y_train_unit[TARGET_COL] == 0).sum()
print(f'Unit binary 분포: pos={n_pos:,} ({n_pos/(n_pos+n_neg):.1%}), neg={n_neg:,}')

## 4. Optuna HPO

- **Sampler**: `TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)`
- **Pruner**: `MedianPruner(n_warmup_steps=10)`
- **Timeout**: `TIMEOUT_SEC` (None=무제한)
- anchor enqueue **없음** (1차 1 trial 신뢰도 낮음)

In [ ]:
# study에 박제할 재현성 메타 — 어떤 조건으로 학습됐는지 study DB만 보고도 알 수 있게
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'clf_model_name':   CLF_MODEL_NAME,
    'raw_mode':         True,                   # raw임을 명시 (분석/필터링용 마커)
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': {},                  # raw mode — preprocess.run 미사용 (빈 dict로 박제)
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'n_jobs':           N_JOBS,
    'timeout_sec':      TIMEOUT_SEC,
    'seed':             SEED,
    'sampler':          'TPE seed=None multivariate group',
    'pruner':           f'MedianPruner n_warmup=10',
}

# TPE: multivariate=HP 결합 분포 학습, group=조건부 축 자동 skip, seed=None → run마다 다양성
sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
# MedianPruner: 처음 10 step은 안 자르고, 그 후 중앙값보다 나쁘면 가지치기
pruner  = MedianPruner(n_warmup_steps=10)

# 분류 HPO: die-level로 학습 + unit 평균 확률 → ×E[Y|Y>0] → unit RMSE를 minimize
# objective가 unit RMSE라 "분류 calibration이 RMSE에 미치는 영향"을 직접 본다. val/test RMSE는 매 trial user_attr에 기록
res = hpo.run_clf_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=CLF_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    user_attrs=study_meta,
    sampler=sampler,
    pruner=pruner,
    enqueue_trials=None,         # anchor enqueue 없음 — 분류기 1차 신뢰도 낮아 시작점 없이 wide 탐색
    timeout=TIMEOUT_SEC,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']
y_pos_const = res['y_pos_const']   # E[Y | Y>0] — Stage 2 회귀를 "상수"로 대체한 값 (combine 전까지 이걸로 평가)

print(f'\n[HPO 완료] best train RMSE = {res["best_value"]:.6f}')
print(f'y_pos_const (E[Y|Y>0]) = {y_pos_const:.6f}')
print(f'best_params = {best_params}')

## 5. Best trial 재학습 (K-fold OOF) + die-level prob 캐쳐

In [5]:
# best HP로 5-fold 재학습 → die-level 양성 확률 (OOF / val / test, val·test는 fold 평균)
final = hpo.refit_clf_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=CLF_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
)

def _unit_mean_proba(xs_split, die_proba):
    # die 확률 → unit 평균 확률
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'p': die_proba})
    return df.groupby(KEY_COL, sort=False)['p'].mean()

def _rmse(pred_unit, y_unit_df):
    # 예측 Series를 정답 순서에 맞춰 정렬 후 RMSE
    aligned = pred_unit.loc[y_unit_df.set_index(KEY_COL).index]
    return float(np.sqrt(np.mean((aligned.values - y_unit_df[TARGET_COL].values) ** 2)))

# unit 평균 확률 × E[Y|Y>0] = "회귀를 상수로 본" unit 예측 → 정답과 RMSE (combine 전 단독 평가)
oof_unit_proba  = _unit_mean_proba(xs_train, final['oof_proba_die'])
val_unit_proba  = _unit_mean_proba(xs_val,   final['val_proba_die'])
test_unit_proba = _unit_mean_proba(xs_test,  final['test_proba_die'])

oof_rmse  = _rmse(oof_unit_proba * y_pos_const, ys_input['train'])
val_rmse  = _rmse(val_unit_proba * y_pos_const, ys_input['validation'])
test_rmse = _rmse(test_unit_proba * y_pos_const, ys_input['test'])

print(f'\n[Refit 완료] (clf 단독, prob × y_pos_const)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

[clf refit fold 1/5] tr_units=20949, vl_units=5238, pos_ratio=0.290


[clf refit fold 2/5] tr_units=20949, vl_units=5238, pos_ratio=0.294


[clf refit fold 3/5] tr_units=20950, vl_units=5237, pos_ratio=0.293


[clf refit fold 4/5] tr_units=20950, vl_units=5237, pos_ratio=0.292


[clf refit fold 5/5] tr_units=20950, vl_units=5237, pos_ratio=0.291

[Refit 완료] (clf 단독, prob × y_pos_const)
  OOF  unit RMSE = 0.005717
  val  unit RMSE = 0.005904
  test unit RMSE = 0.008519


## 6. 산출물 저장

In [6]:
# clf 산출물 저장: oof/val/test die.csv(확률) + unit.csv(평균확률·pred) + fold_models.pkl + best_params.json.
# (clf 단독은 후처리 없음 — combine 단계에서 reg 예측과 곱한 뒤 적용)
hpo.save_clf_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    y_pos_const=y_pos_const,
    study_meta=study_meta,
)

# 저장된 파일 목록
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'clf_{CLF_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass

[save_clf_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\clf\xgb 저장 완료 (fold_models.pkl + best_params.json + 6 CSV)
  best_params.json                       9.8 KB
  fold_models.pkl                   57,659.0 KB
  oof_die.csv                        5,340.3 KB
  oof_unit.csv                       1,453.3 KB
  optuna_jh_ts-clf-xgb-002.db          112.0 KB
  test_die.csv                       1,779.5 KB
  test_unit.csv                        484.3 KB
  val_die.csv                        1,779.6 KB
  val_unit.csv                         484.2 KB
